### Implement cross-validation with different folds by designing custom k-fold and stratified split mechanisms, ensuring balanced class representation, and analyzing variance in model performance across iterations.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

df = pd.read_csv("heart_disease.csv")

print("Dataset shape:", df.shape)
print(df.head())
print("Columns:", df.columns.tolist())

target_col = "target"

if target_col not in df.columns:
    raise ValueError(
        f"Target '{target_col}' not found. "
        f"Available columns: {df.columns.tolist()}"
    )

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

valid = y.notna()
X = X.loc[valid].reset_index(drop=True)
y = y.loc[valid].reset_index(drop=True)

y = pd.Series(pd.factorize(y)[0], name=target_col)

print("Features:", X.shape)
print("Target distribution:")
print(y.value_counts())


Dataset shape: (25, 14)
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  
Columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']
Features: (25, 13)
Target distribution:
target
0    16
1     9
Name: count, dtype: int64


In [2]:
def custom_kfold(n_samples, k=5, shuffle=True, random_state=42):
    if k < 2 or k > n_samples:
        raise ValueError("k must be between 2 and n_samples.")

    indices = np.arange(n_samples)

    if shuffle:
        rng = np.random.default_rng(random_state)
        rng.shuffle(indices)

    folds = np.array_split(indices, k)

    for i in range(k):
        val_idx = folds[i]
        train_idx = np.concatenate(
            [folds[j] for j in range(k) if j != i]
        )

        yield train_idx, val_idx


for fold_no, (train_idx, val_idx) in enumerate(
    custom_kfold(len(X), k=5), start=1
):
    print(
        f"Fold {fold_no}: "
        f"Train = {len(train_idx)}, "
        f"Validation = {len(val_idx)}"
    )


Fold 1: Train = 20, Validation = 5
Fold 2: Train = 20, Validation = 5
Fold 3: Train = 20, Validation = 5
Fold 4: Train = 20, Validation = 5
Fold 5: Train = 20, Validation = 5


In [3]:
def custom_stratified_kfold(
    y, k=5, shuffle=True, random_state=42
):
    y = np.asarray(y)
    n_samples = len(y)

    if k < 2 or k > n_samples:
        raise ValueError("k must be between 2 and n_samples.")

    classes, counts = np.unique(y, return_counts=True)

    if counts.min() < k:
        raise ValueError(
            "Each class must have at least k samples "
            "for every validation fold to contain that class."
        )

    rng = np.random.default_rng(random_state)
    fold_parts = [[] for _ in range(k)]

    for cls in classes:
        cls_idx = np.where(y == cls)[0]

        if shuffle:
            rng.shuffle(cls_idx)

        class_folds = np.array_split(cls_idx, k)

        for fold_no, part in enumerate(class_folds):
            fold_parts[fold_no].extend(part.tolist())

    all_indices = np.arange(n_samples)

    for fold_no in range(k):
        val_idx = np.array(fold_parts[fold_no], dtype=int)

        if shuffle:
            rng.shuffle(val_idx)

        train_mask = np.ones(n_samples, dtype=bool)
        train_mask[val_idx] = False
        train_idx = all_indices[train_mask]

        yield train_idx, val_idx


for fold_no, (train_idx, val_idx) in enumerate(
    custom_stratified_kfold(y, k=5), start=1
):
    print(f"\nFold {fold_no}")
    print("Validation size:", len(val_idx))
    print("Class counts:")
    print(pd.Series(y.iloc[val_idx]).value_counts().sort_index())



Fold 1
Validation size: 6
Class counts:
target
0    4
1    2
Name: count, dtype: int64

Fold 2
Validation size: 5
Class counts:
target
0    3
1    2
Name: count, dtype: int64

Fold 3
Validation size: 5
Class counts:
target
0    3
1    2
Name: count, dtype: int64

Fold 4
Validation size: 5
Class counts:
target
0    3
1    2
Name: count, dtype: int64

Fold 5
Validation size: 4
Class counts:
target
0    3
1    1
Name: count, dtype: int64


In [4]:
models = {
    "Logistic Regression": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=2000, random_state=42)
    ),

    "Decision Tree": make_pipeline(
        SimpleImputer(strategy="median"),
        DecisionTreeClassifier(max_depth=5, random_state=42)
    ),

    "Random Forest": make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            n_jobs=-1
        )
    )
}

In [6]:
print("Target class distribution:")
print(y.value_counts())

print("\nMinimum samples in any class:")
print(y.value_counts().min())

Target class distribution:
target
0    16
1     9
Name: count, dtype: int64

Minimum samples in any class:
9


In [7]:
def custom_stratified_kfold(
    y, k=5, shuffle=True, random_state=42
):
    y = np.asarray(y)
    n_samples = len(y)

    if k < 2 or k > n_samples:
        raise ValueError("k must be between 2 and n_samples.")

    classes, counts = np.unique(y, return_counts=True)

    min_class_count = counts.min()

    if min_class_count < k:
        raise ValueError(
            f"Cannot use {k} stratified folds because the "
            f"smallest class has only {min_class_count} samples. "
            f"Use k <= {min_class_count}."
        )

    rng = np.random.default_rng(random_state)
    fold_parts = [[] for _ in range(k)]

    for cls in classes:
        class_idx = np.where(y == cls)[0]

        if shuffle:
            rng.shuffle(class_idx)

        for fold_no, part in enumerate(np.array_split(class_idx, k)):
            fold_parts[fold_no].extend(part.tolist())

    all_indices = np.arange(n_samples)

    for fold_no in range(k):
        val_idx = np.array(fold_parts[fold_no], dtype=int)

        if shuffle:
            rng.shuffle(val_idx)

        train_mask = np.ones(n_samples, dtype=bool)
        train_mask[val_idx] = False

        yield all_indices[train_mask], val_idx


In [10]:
min_class_count = int(y.value_counts().min())

k_stratified = min(10, min_class_count)

print("Smallest class count:", min_class_count)
print("Stratified folds:", k_stratified)

results_10_kfold = evaluate_custom_cv(
    X, y, models, k=10, method="kfold"
)

results_stratified = evaluate_custom_cv(
    X, y, models, k=k_stratified, method="stratified"
)

all_results = pd.concat(
    [
        results_5_kfold,
        results_5_stratified,
        results_10_kfold,
        results_stratified
    ],
    ignore_index=True
)

print(all_results.head(10))


Smallest class count: 9
Stratified folds: 9
  Method  K                Model  Fold  Train_Size  Validation_Size  Accuracy  \
0  kfold  5  Logistic Regression     1          20                5       0.6   
1  kfold  5  Logistic Regression     2          20                5       1.0   
2  kfold  5  Logistic Regression     3          20                5       1.0   
3  kfold  5  Logistic Regression     4          20                5       1.0   
4  kfold  5  Logistic Regression     5          20                5       0.6   
5  kfold  5        Decision Tree     1          20                5       1.0   
6  kfold  5        Decision Tree     2          20                5       1.0   
7  kfold  5        Decision Tree     3          20                5       0.6   
8  kfold  5        Decision Tree     4          20                5       1.0   
9  kfold  5        Decision Tree     5          20                5       0.6   

   Precision  Recall        F1  
0       0.80     0.6  0.566667 

In [14]:
results_5_kfold = evaluate_custom_cv(
    X, y, models, k=5, method="kfold"
)

results_5_stratified = evaluate_custom_cv(
    X, y, models, k=5, method="stratified"
)

results_10_kfold = evaluate_custom_cv(
    X, y, models, k=10, method="kfold"
)

min_class_count = int(y.value_counts().min())
k_stratified = min(10, min_class_count)

results_stratified = evaluate_custom_cv(
    X, y, models,
    k=k_stratified,
    method="stratified"
)

all_results = pd.concat(
    [
        results_5_kfold,
        results_5_stratified,
        results_10_kfold,
        results_stratified
    ],
    ignore_index=True
)

print("Stratified fold count used:", k_stratified)
display(all_results.head(10))


Stratified fold count used: 9


,Method,K,Model,Fold,Train_Size,Validation_Size,Accuracy,Precision,Recall,F1
0,kfold,5,Logistic Regression,1,20,5,0.6,0.80,0.6,0.566667
1,kfold,5,Logistic Regression,2,20,5,1.0,1.00,1.0,1.000000
2,kfold,5,Logistic Regression,3,20,5,1.0,1.00,1.0,1.000000
3,kfold,5,Logistic Regression,4,20,5,1.0,1.00,1.0,1.000000
4,kfold,5,Logistic Regression,5,20,5,0.6,0.60,0.6,0.600000
5,kfold,5,Decision Tree,1,20,5,1.0,1.00,1.0,1.000000
6,kfold,5,Decision Tree,2,20,5,1.0,1.00,1.0,1.000000
7,kfold,5,Decision Tree,3,20,5,0.6,0.36,0.6,0.450000
8,kfold,5,Decision Tree,4,20,5,1.0,1.00,1.0,1.000000
9,kfold,5,Decision Tree,5,20,5,0.6,0.60,0.6,0.600000
